# Kandinsky 6 SR — minimal example

Runs the packaged video super-resolution pipeline (**KVAE encode → latent upscaler → DiT (tiled) → KVAE decode**) on one video file and saves the result.

Model weights come from the Hugging Face Diffusers bundle below (the SR DiT, the KVAE and the latent upscalers in one repository) and are downloaded into the HF cache on first use. Install the package first: `just setup` in this repository, or `pip install "kandinsky-6-sr"` elsewhere.

The same settings can live in a YAML with an `sr:` section (see `src/kandinsky_sr/configs/sr_release_hf.yaml`) — pass its path to `load_sr_pipeline` instead of an `SRConfig`.

### 1. Config

The model bundle (a Hugging Face repo id or a local copy), upscale factor and device — the same fields as the `sr:` section of a YAML config.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Video, display

from kandinsky_sr.pipeline.config import SRConfig

DEVICE = "cuda:0"
INPUT_VIDEO = Path("inputs/lq.mp4")  # any mp4/mkv; resampled to 24 fps and capped to 5 s (121 frames) below
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SR_CONFIG = SRConfig(
    # The Diffusers bundle: the π-Flow distilled DiT (2 evaluations per tile), the KVAE and the latent upscalers.
    checkpoint_path="kandinskylab/Kandinsky-6.0-VSR-distilled2steps-5s-Diffusers",
    resolution_scale=2,  # 2, 4, or 2.25 (x1.125 pixel pre-upscale + x2 tiling)
    vae_backend="torch",  # "magi" needs the `magi` extra
    device=DEVICE,
)

### 2. Load the input video

Read as a `[T, 3, H, W]` uint8 tensor, then resampled to the model's 24 fps and cut to the 5 s budget (121 frames, on the `1 + 8k` frame grid) — the same preparation `kandy-sr from-video` does.

In [ ]:
from kandinsky_sr.pipeline.video_io import clip_to_aligned_frames, read_video_tchw_uint8, resample_to_target_fps

lq_video, lq_fps = read_video_tchw_uint8(INPUT_VIDEO)
lq_video, OUT_FPS = resample_to_target_fps(lq_video, lq_fps)
lq_video = clip_to_aligned_frames(lq_video)
print(f"input tensor: {tuple(lq_video.shape)} {lq_video.dtype}, source {lq_fps:g} FPS -> {OUT_FPS} FPS")
display(Video(str(INPUT_VIDEO), embed=False, width=640))

### 3. Load the SR pipeline

`load_sr_pipeline` builds the DiT, KVAE and latent upscalers on the device; the warmups below compile the attention kernels and the KVAE decode once, so the first tile does not pay for it.

In [ ]:
from kandinsky_sr.pipeline.components import scale_factor_for
from kandinsky_sr.pipeline.factory import load_sr_pipeline
from kandinsky_sr.pipeline.warmup import clip_base_resolution, compile_vae_decode, warmup

# force=True: the pipeline is built regardless of `sr.enabled` (that flag only gates the k6-embedded route).
sr_pipe = load_sr_pipeline(SR_CONFIG, DEVICE, force=True)
print(f"loaded SR pipeline on {DEVICE}; scale={SR_CONFIG.resolution_scale}")

# One-time compile work, as kandy-sr does: flex/nabla attention kernels, then the
# KVAE decode for the base resolution this clip tiles into.
warmup(sr_pipe.dit, scale_factor_for(sr_pipe), DEVICE)
compile_vae_decode(sr_pipe, DEVICE, bases=[clip_base_resolution(sr_pipe, tuple(lq_video.shape[-2:]))])

### 4. Inference

Tiled SR with a progress bar; the source audio track is carried over into the saved file.

In [ ]:
sr_path = OUT_DIR / f"{INPUT_VIDEO.stem}_sr_{str(SR_CONFIG.resolution_scale).replace('.', 'p')}x.mp4"
with torch.inference_mode():
    sr_result = sr_pipe(
        video=lq_video,
        fps=OUT_FPS,
        show_progress=True,
        source_video=INPUT_VIDEO,  # keeps the source audio track
        save_path=sr_path,
    )
print(f"SR output: {tuple(sr_result.frames.shape)} -> {sr_path}")

In [ ]:
display(Video(str(sr_path), embed=False, width=640))